In [2]:
import pandas as pd
import numpy as np

In [5]:
actions = pd.read_csv("recommendation_actions.csv")

mapping = pd.read_csv("context_action_mapping.csv")

bandit_train = pd.read_csv(
    "train_bandit_interactions.csv"
)

bandit_test = pd.read_csv(
    "test_bandit_interactions.csv"
)

In [6]:
print("Actions:", actions.shape)
print("Mapping:", mapping.shape)
print("Bandit train:", bandit_train.shape)
print("Bandit test:", bandit_test.shape)

Actions: (10, 4)
Mapping: (10, 4)
Bandit train: (63840, 17)
Bandit test: (16000, 17)


In [7]:
actions.head()

,action_id,recommendation,category,evidence_based_trigger
0,A01,Adopt digital payments,Digital,Low digital adoption and no mobile-money use
1,A02,Improve online and customer reach,Digital/Marketing,Low customer reach or low digital adoption
2,A03,Explore appropriate business finance,Finance,Finance constraint or no bank account
3,A04,Improve cash-flow and credit management,Finance,Credit buying/selling and low financial resili...
4,A05,Strengthen management practices,Management,Low management score or no profit/loss tracking


In [8]:
actions.columns.tolist()

['action_id', 'recommendation', 'category', 'evidence_based_trigger']

In [9]:
actions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   action_id               10 non-null     object
 1   recommendation          10 non-null     object
 2   category                10 non-null     object
 3   evidence_based_trigger  10 non-null     object
dtypes: object(4)
memory usage: 452.0+ bytes


In [10]:
actions.isnull().sum()

,0
action_id,0
recommendation,0
category,0
evidence_based_trigger,0


In [11]:
actions.duplicated().sum()

np.int64(0)

In [12]:
actions["action_id"].nunique()

10

In [13]:
actions["action_id"].value_counts()

,count
action_id,
A01,1
A02,1
A03,1
A04,1
A05,1
A06,1
A07,1
A08,1
A09,1


In [14]:
mapping.head()

,context_rule,action_id,priority,rationale
0,digital_adoption_score < 0.5,A01,High,Low observed technology adoption
1,customer_reach_constraint == 1,A02,High,Low digital/customer-reach capability
2,finance_constraint == 1,A03,High,Weak access to formal finance
3,sells_on_credit == 1 or buys_on_credit == 1,A04,Medium,Credit exposure
4,management_score_proxy < 40,A05,High,Low management-practice score


In [15]:
mapping.columns.tolist()

['context_rule', 'action_id', 'priority', 'rationale']

In [16]:
mapping.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   context_rule  10 non-null     object
 1   action_id     10 non-null     object
 2   priority      10 non-null     object
 3   rationale     10 non-null     object
dtypes: object(4)
memory usage: 452.0+ bytes


In [17]:
mapping.isnull().sum()

,0
context_rule,0
action_id,0
priority,0
rationale,0


In [18]:
set(mapping["action_id"]) - set(actions["action_id"])

set()

In [19]:
bandit_train.head()

,interaction_id,round,business_id,city,sector,employee_count,digital_adoption_score,finance_constraint,customer_reach_constraint,management_constraint,profit_last_month,action_id,context_action_fit,accepted,simulated_outcome,roi_proxy,reward
0,I000001,1,V2V0001,Hyderabad,Other Services,2,0.5,1,1,0,1,A01,0.35,1,0.175,0.247,0.445
1,I000002,1,V2V0001,Hyderabad,Other Services,2,0.5,1,1,0,1,A02,1.00,1,0.655,0.757,0.860
2,I000003,1,V2V0001,Hyderabad,Other Services,2,0.5,1,1,0,1,A03,1.00,1,0.991,1.000,0.997
3,I000004,1,V2V0001,Hyderabad,Other Services,2,0.5,1,1,0,1,A04,0.85,1,0.966,0.910,0.931
4,I000005,1,V2V0001,Hyderabad,Other Services,2,0.5,1,1,0,1,A05,0.40,0,0.650,0.749,0.427


In [20]:
bandit_train.columns.tolist()

['interaction_id',
 'round',
 'business_id',
 'city',
 'sector',
 'employee_count',
 'digital_adoption_score',
 'finance_constraint',
 'customer_reach_constraint',
 'management_constraint',
 'profit_last_month',
 'action_id',
 'context_action_fit',
 'accepted',
 'simulated_outcome',
 'roi_proxy',
 'reward']

In [21]:
bandit_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63840 entries, 0 to 63839
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   interaction_id             63840 non-null  object 
 1   round                      63840 non-null  int64  
 2   business_id                63840 non-null  object 
 3   city                       63840 non-null  object 
 4   sector                     63840 non-null  object 
 5   employee_count             63840 non-null  int64  
 6   digital_adoption_score     63840 non-null  float64
 7   finance_constraint         63840 non-null  int64  
 8   customer_reach_constraint  63840 non-null  int64  
 9   management_constraint      63840 non-null  int64  
 10  profit_last_month          63840 non-null  int64  
 11  action_id                  63840 non-null  object 
 12  context_action_fit         63840 non-null  float64
 13  accepted                   63840 non-null  int

In [22]:
bandit_train.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
interaction_id,63840,63840,I079824,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
round,63840.0,NaN,NaN,NaN,4.5,2.291306,1.0,2.75,4.5,6.25,8.0
business_id,63840,798,V2V0998,80,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,63840,8,Ludhiana,8800,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sector,63840,3,Retail,26000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employee_count,63840.0,NaN,NaN,NaN,1.672932,0.682434,1.0,1.0,2.0,2.0,5.0
digital_adoption_score,63840.0,NaN,NaN,NaN,0.344737,0.296458,0.0,0.0,0.2,0.5,1.0
finance_constraint,63840.0,NaN,NaN,NaN,0.844612,0.362277,0.0,1.0,1.0,1.0,1.0
customer_reach_constraint,63840.0,NaN,NaN,NaN,0.948622,0.22077,0.0,1.0,1.0,1.0,1.0
management_constraint,63840.0,NaN,NaN,NaN,0.610276,0.487691,0.0,0.0,1.0,1.0,1.0


In [23]:
bandit_train.isnull().sum()

,0
interaction_id,0
round,0
business_id,0
city,0
sector,0
employee_count,0
digital_adoption_score,0
finance_constraint,0
customer_reach_constraint,0
management_constraint,0
